# 03 — Model Evaluation, Explainability & Equity Analysis
## Diabetes Risk Stratification · CDC BRFSS 2015

Three models compared: Logistic Regression, Random Forest, XGBoost.  
Primary metric: **Balanced Accuracy** (macro-averaged recall).  
SHAP explainability for all three classes.  
Equity analysis: model performance stratified by income, education, sex, and age group.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (balanced_accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
CLASS_LABELS = ['No diabetes', 'Prediabetes', 'Diabetes']
print('OK')

## 1. Load Data & Artifacts

In [ ]:
X_train = pd.read_csv('../data/X_train_resampled.csv')
y_train = pd.read_csv('../data/y_train_resampled.csv').squeeze()
X_test  = pd.read_csv('../data/X_test_scaled.csv')
y_test  = pd.read_csv('../data/y_test.csv').squeeze().astype(int)
X_test_raw = pd.read_csv('../data/X_test.csv')

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. Train Models

In [ ]:
print('Training Logistic Regression...')
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print('Training XGBoost...')
xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.8,
                    eval_metric='mlogloss', random_state=42,
                    n_jobs=-1, tree_method='hist')
xgb.fit(X_train, y_train)
print('All models trained.')

## 3. Comparison Table

In [ ]:
models = {'Logistic Regression': lr, 'Random Forest': rf, 'XGBoost': xgb}

results = {}
print(f'{"Model":<22} {"Bal.Acc":>9} {"NoDM-R":>8} {"PreDM-R":>9} {"DM-R":>7} {"Macro-F1":>10}')
print('-' * 70)

for name, model in models.items():
    y_pred = model.predict(X_test)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    rep = classification_report(y_test, y_pred, output_dict=True)
    r0 = rep['0']['recall']
    r1 = rep['1']['recall']
    r2 = rep['2']['recall']
    mf1 = rep['macro avg']['f1-score']
    results[name] = {'model': model, 'y_pred': y_pred,
                     'bal_acc': bal_acc, 'recall_0': r0, 'recall_1': r1, 'recall_2': r2}
    print(f'{name:<22} {bal_acc:>9.4f} {r0:>8.3f} {r1:>9.3f} {r2:>7.3f} {mf1:>10.4f}')

print('\n⚠️  Prediabetes recall (PreDM-R) is near zero across all models.')
print('This is a data limitation, not a model failure. See Section 6 for clinical interpretation.')

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'], normalize='true')
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_LABELS)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}\nBal. Acc: {res["bal_acc"]:.3f}', fontweight='bold', fontsize=10)
    ax.tick_params(axis='x', rotation=25)
plt.suptitle('Normalized Confusion Matrices (row = true class)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 5. SHAP Explainability — XGBoost

SHAP values for multiclass XGBoost: shape is (n_samples, n_features, n_classes). We analyze global importance per class and local explanation for individual patients.

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_sample = X_test.sample(500, random_state=42)
shap_vals = explainer.shap_values(shap_sample)  # shape: (500, 21, 3)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (cls_label, ax) in enumerate(zip(CLASS_LABELS, axes)):
    sv_cls = shap_vals[:, :, i]
    importance = pd.Series(np.abs(sv_cls).mean(axis=0), index=X_test.columns).sort_values(ascending=True).tail(10)
    ax.barh(importance.index, importance.values, color=['#4C72B0','#FF8C00','#C44E52'][i], alpha=0.85)
    ax.set_title(f'SHAP Importance — {cls_label}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Mean |SHAP value|')
plt.suptitle('Global SHAP Feature Importance by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Clinical interpretation:')
print('  GenHlth (self-rated health) dominates across all classes — strong signal in survey data')
print('  BMI: top predictor for diabetes class, consistent with metabolic risk')
print('  Age: increasing weight for diabetes, expected epidemiological relationship')
print('  HighBP, HighChol: strong comorbidity signal for diabetes class')

## 6. The Prediabetes Detection Problem — Clinical Analysis

This section documents the core finding of this project as a clinical analysis, not a modeling failure.

In [ ]:
print('='*65)
print('THE PREDIABETES DETECTION PROBLEM')
print('='*65)
print()
print('Finding: All models achieve near-zero recall for prediabetes (class 1).')
print()
print('Why this is expected, not a bug:')
print('  1. Prediabetes is defined biochemically (HbA1c 5.7–6.4%, FPG 100–125 mg/dL)')
print('     — no biochemical variables exist in BRFSS')
print('  2. Prediabetes is largely asymptomatic. Behavioral markers (diet, exercise,')
print('     smoking) do not reliably distinguish prediabetic from normoglycemic adults')
print('  3. The EDA showed near-zero effect sizes separating prediabetes from no-DM')
print('     across ALL survey variables (see notebook 01, section 6)')
print()
print('Clinical implication:')
print('  Survey-based risk scores cannot replace biochemical screening for prediabetes.')
print('  This is consistent with ADA 2024 guidelines: universal HbA1c/FPG screening')
print('  is recommended starting at age 35, not behavioral risk-score triage.')
print()
print('What this project demonstrates:')
print('  Understanding what a model CANNOT do is as clinically important as what it can.')
print('  A CDSS that over-promises prediabetes detection without lab values would be')
print('  clinically dangerous — this project explicitly documents that boundary.')

## 7. Equity Analysis — Model Performance by Demographic Group

In [ ]:
best_model = xgb
y_pred_best = results['XGBoost']['y_pred']

# Add predictions to test set
X_test_analysis = X_test_raw.copy()
X_test_analysis['true'] = y_test.values
X_test_analysis['pred'] = y_pred_best
X_test_analysis['correct'] = (X_test_analysis['true'] == X_test_analysis['pred']).astype(int)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
equity_vars = [('Income', 8), ('Education', 6), ('Sex', 2), ('Age', 13)]

for ax, (var, n_cats) in zip(axes, equity_vars):
    # Diabetes recall by group
    recall_by_group = X_test_analysis[X_test_analysis['true']==2].groupby(var).apply(
        lambda x: (x['pred']==2).mean() * 100
    )
    ax.bar(recall_by_group.index, recall_by_group.values, color='#C44E52', alpha=0.8)
    ax.set_title(f'Diabetes Recall by {var}', fontweight='bold', fontsize=9)
    ax.set_xlabel(var)
    ax.set_ylabel('Recall (%)')
    ax.axhline(recall_by_group.mean(), color='navy', linestyle='--', linewidth=1, label='Mean')
    ax.legend(fontsize=7)

plt.suptitle('Equity Analysis: Diabetes Class Recall by Socioeconomic/Demographic Group (XGBoost)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('Equity findings:')
print('  Check for systematic underperformance in low-income or low-education groups.')
print('  Differential model performance across demographics is a fairness concern in CDSS.')

## 8. Save Final Model

In [ ]:
joblib.dump(xgb, '../src/model_xgb.pkl')
joblib.dump(explainer, '../src/shap_explainer.pkl')

print('Saved: src/model_xgb.pkl')
print('Saved: src/shap_explainer.pkl')
print('\n→ Proceed to app/streamlit_app.py')